# 01 · EDA — Alice's Wonderland reasoning puzzles

Confirm the six puzzle categories, look at length distributions, sample a few
puzzles per category to see what the model is expected to do.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import polars as pl
from src.data import load_train, load_test

train = load_train()
test = load_test()
print('train shape:', train.shape)
print('test shape :', test.shape)
train.head(3)

## Category distribution

In [ ]:
train.group_by('category').agg(pl.len().alias('count')).sort('count', descending=True)

## Prompt and answer length distributions

In [ ]:
stats = train.with_columns([
    pl.col('prompt').str.len_chars().alias('prompt_chars'),
    pl.col('answer').str.len_chars().alias('answer_chars'),
])

stats.group_by('category').agg([
    pl.col('prompt_chars').mean().round(0).alias('prompt_mean'),
    pl.col('prompt_chars').max().alias('prompt_max'),
    pl.col('answer_chars').mean().round(1).alias('answer_mean'),
    pl.col('answer_chars').min().alias('answer_min'),
    pl.col('answer_chars').max().alias('answer_max'),
]).sort('category')

## One sample per category

In [ ]:
for cat in sorted(train['category'].unique()):
    row = train.filter(pl.col('category') == cat).head(1).to_dicts()[0]
    print('=' * 80)
    print(f'CATEGORY: {cat}   id={row["id"]}')
    print('=' * 80)
    print(row['prompt'])
    print()
    print(f'ANSWER: {row["answer"]}')
    print()

## Quick sanity check on the local metric

In [ ]:
from src.metric import is_correct, extract_answer, score

examples = [
    ('The result is therefore \\boxed{42}.', '42', True),
    ('I think the answer might be 41.999, so \\boxed{42.005}', '42', True),  # within 1e-2 rel tol
    ('\\boxed{wrong}', 'right', False),
    ('No box, just an answer of 7', '7', True),  # fallback to last number
]
for text, target, expected in examples:
    got = is_correct(text, target)
    print(f'extract={extract_answer(text)!r:>15}  target={target!r:>10}  ok={got}  expected={expected}')